In [2]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
iris = fetch_ucirepo(id=53) 
  
# data (as pandas dataframes) 
X = iris.data.features 
y = iris.data.targets 
  
# metadata 
print(iris.metadata) 
  
# variable information 
print(iris.variables) 


{'uci_id': 53, 'name': 'Iris', 'repository_url': 'https://archive.ics.uci.edu/dataset/53/iris', 'data_url': 'https://archive.ics.uci.edu/static/public/53/data.csv', 'abstract': 'A small classic dataset from Fisher, 1936. One of the earliest known datasets used for evaluating classification methods.\n', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 150, 'num_features': 4, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1936, 'last_updated': 'Tue Sep 12 2023', 'dataset_doi': '10.24432/C56C76', 'creators': ['R. A. Fisher'], 'intro_paper': {'ID': 191, 'type': 'NATIVE', 'title': 'The Iris data set: In search of the source of virginica', 'authors': 'A. Unwin, K. Kleinman', 'venue': 'Significance, 2021', 'year': 2021, 'journal': 'Significance, 2021', 'DOI': '1740-9713.01589', 'URL': 'https://www.semanticscholar.org

In [ ]:
'''
1° Passo - Matriz W
  Para cada linha, sortear 3 valores entre 0 e 1, cuja soma deles dá 1 (100%)
  Normalizar (??????????????????????????????????????????)
--> Teremos um peso para cada cluster (coluna --> 3 C1, C2, C3) para cada amostra (linha --> 150)

2° Passo - Calcular o centróide dos cluster
  Pra cada coluna (cluster)
    soma os valores da linha
    Aplica fórmula
    Peso_centroide_x = soma (vetor_linha * peso_linha --> vet*0.7) / som dos pesos (todas linhas)
          cj         =    ∑   xi        *   wij                   /    ∑   wij

    xi = [sepal length, sepal width, petal length, petal width]
    wij = somatória dos pesos
    cj = vetor com 4 entradas --> centroide pra cada variável     [c1, c2, c3, c4]

3° Passo - Atualizar pesos
  Pra cada linha
    Calcula o peso pra cada cluster
    wi1 = xxxxx <-- dist_euclidiana(Xi - c1) / dist_euclidiana(Xi - c1) + dist_euclidiana(Xi - c2) + dist_euclidiana(Xi - c3)
    wi2 = xxxxx
    wi3 = xxxxx


'''

In [3]:
# 1° Passo - Montar Matriz W com sorteio

import random
import pandas as pd
import numpy as np

# Amostra C1 C2 C3
W = []

# Teremos que fazer o sorteio de 3 valores 150 vezes (tamanho da iris)
for linha in range(len(X)):
    # Gerando 3 números aleatórios
    prob = [random.random() for _ in range(3)]

    # Transformando em entre 0 e 1 (percentual)
    prob_normalizados = [x / sum(prob) for x in prob]
    
    W.append(prob_normalizados)
    
W = pd.DataFrame(W, columns=['C1', 'C2', 'C3'])
W 

,C1,C2,C3
0,0.198362,0.406401,0.395237
1,0.193557,0.493358,0.313084
2,0.529476,0.274414,0.196110
3,0.533322,0.019946,0.446732
4,0.411707,0.415200,0.173093
...,...,...,...
145,0.046260,0.437075,0.516665
146,0.215307,0.747182,0.037511
147,0.618793,0.305095,0.076113
148,0.166232,0.649941,0.183826


In [ ]:
# 2° Passo - Calcular o centróide dos cluster

# Criando o campo xi com todos os dados direto no df
X['xi'] = list(X[['sepal length', 'sepal width', 'petal length', 'petal width']].values)


def calcular_centroides(X, W):
    centroides = []

    # Para cada coluna, calcularemos o centroide
    for col in W.columns:
        sum_xi_wij = sum(xi * peso for xi, peso in zip(X['xi'], W[col]))
        sum_wij = W[col].sum()
        
        centroides.append(sum_xi_wij / sum_wij)

    return centroides

In [7]:
# 3° Passo - Atualizar pesos

# Função de distância Euclidiana
def dist(a, b):
    return np.linalg.norm(a - b)

# Função para atualizar os pesos
def att_pesos(centroides, X):
    m = 2 
    novos_pesos = []
    # Pra cada linha, atualizaremos os pesos
    for _, linha in X.iterrows():
        # Extraindo a linha
        xi = linha['xi']
        
        # Calculando a distância de cada centroide para essa linha
        distancias = [dist(xi, c) for c in centroides]
        
        probabilidade_att = []
        for d in distancias:
            # Se a distância for 0, o peso é 1
            if d == 0:
                probabilidade_att.append(1.0)
            else:
                soma_razões = sum([(d / d_2)**(2/(m-1)) for d_2 in distancias])
                probabilidade_att.append(1 / soma_razões)
                
        novos_pesos.append(probabilidade_att)

    # Atualizando a matriz W com os novos pesos calculados
    W = pd.DataFrame(novos_pesos, columns=['C1', 'C2', 'C3'])
    return W

In [8]:
# Colocar isso em um loop
# Critério de parada?

for i in range(20):
   centroides = calcular_centroides(X, W)
   W = att_pesos(centroides, X)
   print(centroides, '\n')

[array([3.92453288, 1.27027111, 5.91647509, 3.04298472]), array([3.5733888 , 1.12909833, 5.78553696, 3.09446822]), array([3.78353566, 1.19879198, 5.82994805, 3.0236587 ])] 

[array([4.00623203, 1.3002429 , 5.93980357, 3.02576887]), array([3.44057932, 1.0694426 , 5.72770357, 3.10048146]), array([3.77598327, 1.20460733, 5.84255521, 3.0427925 ])] 

[array([4.13053581, 1.35107911, 5.98521034, 3.00828312]), array([3.23289798, 0.9834211 , 5.64835878, 3.12694378]), array([3.78949814, 1.21108516, 5.85006484, 3.04277124])] 

[array([4.29538577, 1.42041243, 6.05084281, 2.99011638]), array([2.90925799, 0.84880943, 5.527545  , 3.17387345]), array([3.82726195, 1.22604172, 5.85854292, 3.029004  ])] 

[array([4.50631606, 1.51141456, 6.14037252, 2.97157457]), array([2.48155386, 0.66994601, 5.37003618, 3.2407804 ]), array([3.93072085, 1.26539351, 5.87921165, 2.99123026])] 

[array([4.75808443, 1.62314466, 6.25325112, 2.9549319 ]), array([2.07584686, 0.49924158, 5.22277028, 3.30738826]), array([4.145257

In [ ]:
# Pendências:
# Como escolhe o m??
# Normalizar? Isso é no sorteio?  <-- confirmar


# Pensar e implementar um critério de parada
# Classificar classe pra cada amostras 
# Plotar
# Talvez fazer um comparativo com os dados reais

# Duvida do passo 3:
# Pega a db, filtra pra virgínica e versicolor e roda tudo dnv só que com 2 clusters?
# o que é demonstrar a pertinência intermediária?

# É necessário tornar o código universal? Tipo fazer uma biblioteca?


In [9]:
W

,C1,C2,C3
0,0.007364,0.981040,0.011596
1,0.013474,0.964936,0.021591
2,0.016288,0.958156,0.025555
3,0.017109,0.955486,0.027405
4,0.009425,0.975828,0.014747
...,...,...,...
145,0.864529,0.011300,0.124171
146,0.662711,0.013385,0.323904
147,0.976792,0.001357,0.021852
148,0.772618,0.020962,0.206420
